# Super Voz - Kaggle

Execute as celulas em ordem com GPU ligada. O notebook baixa `warllem/Super_voz` do Hugging Face usando o token salvo nos Kaggle Secrets, seleciona o melhor checkpoint pelos logs/metrica e abre uma interface para gerar WAV.


In [ ]:
# 1) Conferir GPU e token do Hugging Face nos Kaggle Secrets
import os
import torch

print('CUDA disponivel:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    found = False
    for name in ('HF_TOKEN', 'HUGGINGFACE_TOKEN', 'HUGGING_FACE_HUB_TOKEN'):
        try:
            value = secrets.get_secret(name)
            if value:
                os.environ[name] = value
                print('Secret encontrado:', name)
                found = True
                break
        except Exception:
            pass
    if not found:
        print('Aviso: nenhum secret HF_TOKEN/HUGGINGFACE_TOKEN/HUGGING_FACE_HUB_TOKEN encontrado.')
except Exception as exc:
    print('Aviso: Kaggle Secrets indisponivel neste ambiente:', exc)


In [ ]:
# 2) Instalar dependencias minimas para baixar e detectar o pacote
import subprocess
import sys

def run(command, check=True):
    print('Executando:', ' '.join(command))
    return subprocess.run(command, check=check, text=True)

run(['apt-get', '-qq', 'update'])
run(['apt-get', '-qq', 'install', '-y', 'espeak-ng', 'ffmpeg'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'huggingface_hub>=0.31.0', 'hf_xet>=1.1.0', 'pyyaml>=6.0.0'])
print('Dependencias minimas instaladas.')


In [ ]:
# 3) Criar o modulo do conversor dentro do Kaggle
from pathlib import Path
import sys

MODULE_CODE = 'from __future__ import annotations\n\nimport inspect\nimport os\nimport re\nimport subprocess\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Iterable\n\n\nHF_REPO_ID = "warllem/Super_voz"\nMODEL_ROOT = Path("/kaggle/working/Super_voz")\nOUTPUT_DIR = Path("/kaggle/working/audios_gerados")\nREFERENCE_CANDIDATES = (\n    "data_reference/referencia_voz.wav",\n    "referencia_voz.wav",\n    "audio_warllem_ref_fonemas_ptbr.wav",\n    "audio_warllem_ref_fonemas.wav",\n    "audio_warllem_ref_texto.wav",\n)\n\n\n@dataclass(frozen=True)\nclass ModelBundle:\n    engine: str\n    model_path: Path\n    config_path: Path | None = None\n    reference_audio_path: Path | None = None\n\n\ndef run_command(command: list[str], input_text: str | None = None) -> None:\n    subprocess.run(command, input=input_text, text=True, check=True)\n\n\ndef get_kaggle_secret(name: str) -> str | None:\n    try:\n        from kaggle_secrets import UserSecretsClient\n\n        value = UserSecretsClient().get_secret(name)\n        return value or None\n    except Exception:\n        return None\n\n\ndef get_hf_token() -> str | None:\n    for name in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):\n        value = os.environ.get(name) or get_kaggle_secret(name)\n        if value:\n            return value\n    return None\n\n\ndef download_hf_repo(\n    repo_id: str = HF_REPO_ID,\n    output_dir: Path = MODEL_ROOT,\n    token: str | None = None,\n) -> Path:\n    from huggingface_hub import snapshot_download\n\n    token = token or get_hf_token()\n    output_dir.mkdir(parents=True, exist_ok=True)\n    snapshot_download(\n        repo_id=repo_id,\n        repo_type="model",\n        local_dir=str(output_dir),\n        local_dir_use_symlinks=False,\n        token=token,\n    )\n    return output_dir\n\n\ndef iter_files(root: Path) -> Iterable[Path]:\n    if not root.exists():\n        return []\n    return sorted(path for path in root.rglob("*") if path.is_file())\n\n\ndef print_file_report(root: Path) -> None:\n    files = list(iter_files(root))\n    if not files:\n        print("Nenhum arquivo encontrado.")\n        return\n\n    print("Arquivos encontrados:")\n    for path in files:\n        size_mb = path.stat().st_size / 1024 / 1024\n        print(f"- {path} ({size_mb:.1f} MB)")\n\n\ndef parse_key_value_file(path: Path) -> dict[str, str]:\n    data: dict[str, str] = {}\n    if not path.exists():\n        return data\n    for line in path.read_text(encoding="utf-8", errors="ignore").splitlines():\n        if "=" not in line:\n            continue\n        key, value = line.split("=", 1)\n        data[key.strip()] = value.strip()\n    return data\n\n\ndef parse_validation_losses_from_log(log_path: Path) -> list[tuple[int, float]]:\n    if not log_path.exists():\n        return []\n\n    results: list[tuple[int, float]] = []\n    current_epoch = -1\n    epoch_pattern = re.compile(r"Epoch\\s+\\[(\\d+)/\\d+\\]")\n    val_pattern = re.compile(r"Validation loss:\\s*([0-9.]+)", re.IGNORECASE)\n\n    for line in log_path.read_text(encoding="utf-8", errors="ignore").splitlines():\n        epoch_match = epoch_pattern.search(line)\n        if epoch_match:\n            current_epoch = int(epoch_match.group(1))\n        val_match = val_pattern.search(line)\n        if val_match and current_epoch >= 0:\n            results.append((current_epoch, float(val_match.group(1))))\n    return results\n\n\ndef find_training_logs(root: Path) -> list[Path]:\n    patterns = ("*.log", "*log*.txt", "events.out.tfevents.*")\n    logs: list[Path] = []\n    for pattern in patterns:\n        logs.extend(root.rglob(pattern))\n    return sorted(set(path for path in logs if path.is_file()))\n\n\ndef best_epoch_from_logs(root: Path) -> tuple[int, float] | None:\n    best: tuple[int, float] | None = None\n    preferred_logs = [path for path in (root / "docs" / "train.log", root / "model" / "train.log") if path.exists()]\n    logs = preferred_logs or find_training_logs(root)\n    for log_path in logs:\n        if log_path.name.startswith("events.out.tfevents"):\n            continue\n        for epoch, loss in parse_validation_losses_from_log(log_path):\n            if best is None or loss < best[1]:\n                best = (epoch, loss)\n    return best\n\n\ndef checkpoint_epoch(path: Path) -> int:\n    patterns = (\n        r"epoch_2nd_(\\d+).*\\.pth$",\n        r"epoch_(\\d+).*\\.pth$",\n    )\n    for pattern in patterns:\n        match = re.search(pattern, path.name)\n        if match:\n            return int(match.group(1))\n    return -1\n\n\ndef select_checkpoint_path(root: Path) -> Path | None:\n    pth_files = sorted(root.rglob("*.pth"))\n    if not pth_files:\n        return None\n\n    best_metric = parse_key_value_file(root / "model" / "best_metric.txt")\n    if best_metric:\n        source_checkpoint = best_metric.get("source_checkpoint")\n        candidates = [root / "model" / "best_model.pth"]\n        if source_checkpoint:\n            candidates.append(root / "model" / source_checkpoint)\n        for candidate in candidates:\n            if candidate.is_file():\n                print("Melhor checkpoint pelo best_metric.txt:", candidate)\n                print("Metricas:", best_metric)\n                return candidate\n\n    best_from_log = best_epoch_from_logs(root)\n    if best_from_log:\n        best_epoch, best_loss = best_from_log\n        epoch_matches = [path for path in pth_files if checkpoint_epoch(path) == best_epoch]\n        if epoch_matches:\n            selected = sorted(epoch_matches)[0]\n            print(f"Melhor checkpoint pelo train.log: {selected} (validation_loss={best_loss:.3f})")\n            return selected\n\n    latest_name_path = root / "model" / "latest_checkpoint.txt"\n    if latest_name_path.exists():\n        latest_name = latest_name_path.read_text(encoding="utf-8", errors="ignore").strip()\n        for candidate in (root / "model" / "latest_checkpoint.pth", root / "model" / latest_name):\n            if candidate.is_file():\n                print("Checkpoint latest selecionado:", candidate)\n                return candidate\n\n    preferred_names = ("best_model.pth", "latest_checkpoint.pth")\n    for name in preferred_names:\n        matches = [path for path in pth_files if path.name == name]\n        if matches:\n            return sorted(matches)[0]\n\n    epoch_files = [path for path in pth_files if checkpoint_epoch(path) >= 0]\n    if epoch_files:\n        return sorted(epoch_files, key=lambda path: (checkpoint_epoch(path), str(path)), reverse=True)[0]\n\n    return pth_files[0]\n\n\ndef nearest_config_for_checkpoint(checkpoint: Path, root: Path, patterns: tuple[str, ...]) -> Path | None:\n    candidates: list[Path] = []\n    for pattern in patterns:\n        candidates.extend(root.rglob(pattern))\n    candidates = sorted(set(candidates))\n    if not candidates:\n        return None\n\n    same_folder = [path for path in candidates if path.parent == checkpoint.parent]\n    if same_folder:\n        return same_folder[0]\n\n    parent_folder = [path for path in candidates if checkpoint.parent.is_relative_to(path.parent)]\n    if parent_folder:\n        return sorted(parent_folder, key=lambda path: len(path.parts), reverse=True)[0]\n\n    model_config = root / "model" / "config.yml"\n    if model_config.exists() and model_config in candidates:\n        return model_config\n    return candidates[0]\n\n\ndef score_audio_reference_line(line: str) -> float | None:\n    metric_patterns = (\n        r"(?:score|similarity|mos|qualidade|quality)\\s*[:=]\\s*([0-9.]+)",\n        r"(?:loss|erro|error|wer|cer)\\s*[:=]\\s*([0-9.]+)",\n    )\n    for pattern in metric_patterns:\n        match = re.search(pattern, line, re.IGNORECASE)\n        if not match:\n            continue\n        value = float(match.group(1))\n        if re.search(r"loss|erro|error|wer|cer", pattern, re.IGNORECASE):\n            value = -value\n        return value\n    return None\n\n\ndef select_reference_audio_from_logs(root: Path) -> Path | None:\n    wav_pattern = re.compile(r"([\\w./-]+\\.wav)", re.IGNORECASE)\n    best: tuple[float, Path] | None = None\n\n    for log_path in find_training_logs(root):\n        if log_path.name.startswith("events.out.tfevents"):\n            continue\n        for line in log_path.read_text(encoding="utf-8", errors="ignore").splitlines():\n            score = score_audio_reference_line(line)\n            if score is None:\n                continue\n            for raw_path in wav_pattern.findall(line):\n                candidate = (root / raw_path).resolve()\n                if candidate.exists() and (best is None or score > best[0]):\n                    best = (score, candidate)\n\n    if best:\n        print(f"Audio de referencia escolhido pelo log: {best[1]} (score={best[0]:.4f})")\n        return best[1]\n    return None\n\n\ndef select_reference_audio(root: Path) -> Path | None:\n    from_log = select_reference_audio_from_logs(root)\n    if from_log:\n        return from_log\n\n    for relative_path in REFERENCE_CANDIDATES:\n        candidate = root / relative_path\n        if candidate.exists():\n            print("Audio de referencia escolhido:", candidate)\n            return candidate\n\n    val_list = root / "data_reference" / "val_list.txt"\n    if val_list.exists():\n        for line in val_list.read_text(encoding="utf-8", errors="ignore").splitlines():\n            raw_path = line.split("|", 1)[0].strip()\n            candidate = root / "data_reference" / raw_path\n            if candidate.exists() and candidate.suffix.lower() == ".wav":\n                print("Audio de referencia escolhido pela val_list:", candidate)\n                return candidate\n\n    wav_files = sorted((root / "data_reference").rglob("*.wav")) if (root / "data_reference").exists() else []\n    if wav_files:\n        print("Audio de referencia escolhido pelo primeiro WAV encontrado:", wav_files[0])\n        return wav_files[0]\n\n    return None\n\n\ndef find_coqui_bundle(root: Path) -> ModelBundle | None:\n    model_path = select_checkpoint_path(root)\n    if not model_path:\n        return None\n\n    config_path = nearest_config_for_checkpoint(model_path, root, ("config.json",))\n    if not config_path:\n        return None\n\n    return ModelBundle("coqui", model_path, config_path, select_reference_audio(root))\n\n\ndef find_styletts2_bundle(root: Path) -> ModelBundle | None:\n    model_path = select_checkpoint_path(root)\n    if not model_path:\n        return None\n\n    config_path = nearest_config_for_checkpoint(model_path, root, ("*.yml", "*.yaml"))\n    if not config_path:\n        return None\n\n    config_text = config_path.read_text(encoding="utf-8", errors="ignore")\n    styletts2_markers = ("ASR_config", "PLBERT_dir", "model_params", "preprocess_params")\n    if not all(marker in config_text for marker in styletts2_markers):\n        return None\n\n    return ModelBundle("styletts2", model_path, config_path, select_reference_audio(root))\n\n\ndef find_piper_bundle(root: Path) -> ModelBundle | None:\n    onnx_files = sorted(root.rglob("*.onnx"))\n    if not onnx_files:\n        return None\n    return ModelBundle("piper", onnx_files[0], reference_audio_path=select_reference_audio(root))\n\n\ndef detect_model_bundle(root: Path) -> ModelBundle:\n    styletts2 = find_styletts2_bundle(root)\n    if styletts2:\n        return styletts2\n\n    coqui = find_coqui_bundle(root)\n    if coqui:\n        return coqui\n\n    piper = find_piper_bundle(root)\n    if piper:\n        return piper\n\n    pth_files = sorted(root.rglob("*.pth"))\n    if pth_files:\n        names = ", ".join(path.name for path in pth_files)\n        raise RuntimeError(\n            "Encontrei arquivo .pth, mas nao encontrei configuracao suportada. "\n            f"Arquivos .pth encontrados: {names}"\n        )\n\n    raise RuntimeError("Nao encontrei modelo suportado em /kaggle/working/Super_voz.")\n\n\ndef patch_torch_load_for_styletts2():\n    import torch\n\n    original_load = torch.load\n\n    def load_with_legacy_checkpoint_support(*args, **kwargs):\n        kwargs.setdefault("weights_only", False)\n        return original_load(*args, **kwargs)\n\n    torch.load = load_with_legacy_checkpoint_support\n    return original_load\n\n\ndef restore_torch_load(original_load) -> None:\n    import torch\n\n    torch.load = original_load\n\n\ndef report_styletts2_auxiliary_paths(config_path: Path) -> None:\n    config_text = config_path.read_text(encoding="utf-8", errors="ignore")\n    relative_paths = re.findall(r"(?:ASR_config|ASR_path|F0_path|PLBERT_dir):\\s*([^,\\n}]+)", config_text)\n    missing_paths = []\n    for relative_path in relative_paths:\n        candidate = config_path.parent / relative_path.strip()\n        if not candidate.exists():\n            missing_paths.append(candidate)\n\n    if missing_paths:\n        print("Aviso: arquivos auxiliares do StyleTTS2 nao encontrados localmente:")\n        for path in missing_paths:\n            print(f"- {path}")\n\n\nclass NeuralVoiceSynthesizer:\n    def __init__(self, bundle: ModelBundle, output_dir: Path = OUTPUT_DIR):\n        self.bundle = bundle\n        self.output_dir = output_dir\n        self.output_dir.mkdir(parents=True, exist_ok=True)\n        self.tts = None\n        self._load()\n\n    def _load(self) -> None:\n        if self.bundle.engine == "coqui":\n            import torch\n            from TTS.api import TTS\n\n            self.tts = TTS(\n                model_path=str(self.bundle.model_path),\n                config_path=str(self.bundle.config_path),\n                progress_bar=True,\n                gpu=torch.cuda.is_available(),\n            )\n            print("Modelo Coqui carregado.")\n            return\n\n        if self.bundle.engine == "piper":\n            print("Modelo Piper pronto para inferencia.")\n            return\n\n        if self.bundle.engine == "styletts2":\n            report_styletts2_auxiliary_paths(self.bundle.config_path)\n            original_torch_load = patch_torch_load_for_styletts2()\n            previous_cwd = Path.cwd()\n            os.chdir(self.bundle.config_path.parent)\n            try:\n                from styletts2 import tts\n\n                self.tts = tts.StyleTTS2(\n                    model_checkpoint_path=str(self.bundle.model_path),\n                    config_path=str(self.bundle.config_path),\n                )\n            finally:\n                os.chdir(previous_cwd)\n                restore_torch_load(original_torch_load)\n            print("Modelo StyleTTS2 carregado.")\n            return\n\n        raise ValueError(f"Engine nao suportada: {self.bundle.engine}")\n\n    @staticmethod\n    def _choose_first(values, preferred: list[str] | None = None):\n        if not values:\n            return None\n        preferred = preferred or []\n        lowered = {str(value).lower(): value for value in values}\n        for item in preferred:\n            if item.lower() in lowered:\n                return lowered[item.lower()]\n        return values[0]\n\n    def _next_wav_path(self) -> Path:\n        index = len(list(self.output_dir.glob("audio_*.wav"))) + 1\n        return self.output_dir / f"audio_{index:04d}.wav"\n\n    def synthesize(self, text: str) -> str:\n        text = (text or "").strip()\n        if not text:\n            raise ValueError("Digite uma frase antes de gerar o audio.")\n\n        wav_path = self._next_wav_path()\n        if self.bundle.engine == "coqui":\n            kwargs = {"text": text, "file_path": str(wav_path)}\n            speakers = getattr(self.tts, "speakers", None)\n            languages = getattr(self.tts, "languages", None)\n            speaker = self._choose_first(speakers)\n            language = self._choose_first(languages, ["pt-br", "pt", "portuguese", "portugues"])\n            if speaker:\n                kwargs["speaker"] = speaker\n            if language:\n                kwargs["language"] = language\n            if self.bundle.reference_audio_path and "speaker_wav" in inspect.signature(self.tts.tts_to_file).parameters:\n                kwargs["speaker_wav"] = str(self.bundle.reference_audio_path)\n            self.tts.tts_to_file(**kwargs)\n            return str(wav_path)\n\n        if self.bundle.engine == "styletts2":\n            kwargs = {"output_wav_file": str(wav_path), "output_sample_rate": 24000}\n            parameters = inspect.signature(self.tts.inference).parameters\n            for name in ("target_voice_path", "reference_audio_path", "speaker_wav"):\n                if self.bundle.reference_audio_path and name in parameters:\n                    kwargs[name] = str(self.bundle.reference_audio_path)\n                    break\n            self.tts.inference(text, **kwargs)\n            return str(wav_path)\n\n        run_command(["piper", "--model", str(self.bundle.model_path), "--output_file", str(wav_path)], input_text=text)\n        return str(wav_path)\n\n\ndef create_gradio_app(synthesizer: NeuralVoiceSynthesizer):\n    import gradio as gr\n\n    def generate(text: str):\n        try:\n            audio_path = synthesizer.synthesize(text)\n            return audio_path, audio_path, f"Audio gerado: {audio_path}"\n        except Exception as exc:\n            return None, None, f"Erro: {exc}"\n\n    with gr.Blocks(title="Super Voz") as demo:\n        gr.Markdown("# Super Voz")\n        text_box = gr.Textbox(label="Frase", placeholder="Digite sua frase e pressione Enter...", lines=1)\n        audio = gr.Audio(label="Audio gerado", type="filepath")\n        download = gr.File(label="Download do WAV")\n        status = gr.Textbox(label="Status", interactive=False)\n        text_box.submit(generate, inputs=text_box, outputs=[audio, download, status]).then(lambda: "", outputs=text_box)\n    return demo\n\n\ndef display_notebook_audio(audio_path: str) -> str:\n    from IPython.display import Audio, FileLink, display\n\n    display(Audio(audio_path))\n    display(FileLink(audio_path, result_html_prefix="Download do WAV: "))\n    return audio_path\n\n\ndef synthesize_for_notebook(synthesizer: NeuralVoiceSynthesizer, text: str) -> str:\n    audio_path = synthesizer.synthesize(text)\n    print(f"Audio gerado: {audio_path}")\n    return display_notebook_audio(audio_path)\n\n\ndef load_synthesizer(download: bool = False) -> NeuralVoiceSynthesizer:\n    root = prepare_model_files(download=download)\n    print_file_report(root)\n\n    bundle = detect_model_bundle(root)\n    print(f"Engine detectada: {bundle.engine}")\n    print(f"Modelo: {bundle.model_path}")\n    if bundle.config_path:\n        print(f"Config: {bundle.config_path}")\n    if bundle.reference_audio_path:\n        print(f"Audio referencia: {bundle.reference_audio_path}")\n\n    return NeuralVoiceSynthesizer(bundle)\n\n\ndef prepare_model_files(download: bool = True) -> Path:\n    if download or not MODEL_ROOT.exists():\n        return download_hf_repo()\n    return MODEL_ROOT\n\n\ndef main(download: bool = False) -> None:\n    synthesizer = load_synthesizer(download=download)\n    demo = create_gradio_app(synthesizer)\n    demo.launch(share=True, debug=True)\n\n\nif __name__ == "__main__":\n    main(download=True)\n'
module_path = Path('/kaggle/working/conversor_voz_kaggle.py')
module_path.write_text(MODULE_CODE, encoding='utf-8')
sys.path.insert(0, '/kaggle/working')
print('Modulo criado em', module_path)


In [ ]:
# 4) Baixar o repositorio do Hugging Face e selecionar checkpoint/audio
from conversor_voz_kaggle import download_hf_repo, detect_model_bundle, print_file_report

model_root = download_hf_repo()
print_file_report(model_root)

bundle = detect_model_bundle(model_root)
print('Engine detectada:', bundle.engine)
print('Modelo:', bundle.model_path)
print('Config:', bundle.config_path)
print('Audio referencia:', bundle.reference_audio_path)


In [ ]:
# 5) Instalar dependencias do motor detectado
BASE_PACKAGES = ['gradio>=4.44.0', 'pydub>=0.25.1', 'soundfile>=0.12.1', 'pyyaml>=6.0.0']
ENGINE_PACKAGES = {
    'styletts2': ['styletts2>=0.1.7'],
    'coqui': ['coqui-tts>=0.27.0'],
    'piper': ['piper-tts>=1.2.0'],
}
packages = BASE_PACKAGES + ENGINE_PACKAGES.get(bundle.engine, [])
run([sys.executable, '-m', 'pip', 'install', '-q', '-U', *packages])

if bundle.engine == 'styletts2':
    run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall', 'numpy==1.26.4'])

print('Dependencias instaladas para engine:', bundle.engine)


In [ ]:
# 6) Carregar a voz uma vez
from conversor_voz_kaggle import load_synthesizer, synthesize_for_notebook, create_gradio_app

synthesizer = load_synthesizer(download=False)
print('Voz carregada. Agora use a proxima celula para gerar e baixar audios.')


In [ ]:
# 7) Gerar um audio com player e link de download direto
# Altere o texto abaixo e execute esta celula sempre que quiser gerar um novo WAV.
texto = 'Digite aqui o texto que voce quer transformar em audio.'

audio_path = synthesize_for_notebook(synthesizer, texto)
audio_path


## Interface opcional

A celula abaixo abre a interface Gradio. Use se quiser digitar pela caixa de texto e ouvir/baixar por botoes. Enquanto essa celula estiver rodando, ela segura o notebook; para executar outras celulas depois, pare a celula do Gradio.


In [ ]:
# 8) Opcional: abrir interface Gradio
demo = create_gradio_app(synthesizer)
demo.launch(share=True, debug=True)
